In [18]:
import sys, os
path = os.path.abspath('../src')
sys.path.insert(0, path)

In [19]:
from gedih3.config import GH3_DEFAULT_DOWNLOAD_DIR, GH3_DEFAULT_SOC_DIR, GEDI_START_DATE
from gedih3.gedidriver import soc_file_tree, dask_h5_merged
from gedih3.gh3builder import h3_index_df

In [34]:
%%timeit
a = gpd.read_parquet('../tmp/singleton.parquet')

330 ms ± 6.46 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [33]:
%%timeit
import geopandas as gpd
f = '/home/tiago/Desktop/repos/GEDI-H3/tmp/tmp/l4c/h3_03=838041fffffffff/'
# Reading with pandas (which can use PyArrow engine)
# df = gpd.read_parquet(f)
# df

# Or reading directly with PyArrow
# table = pq.read_table(f)
# df = table.to_pandas()

# # For partitioned datasets
# dataset = pq.ParquetDataset('/home/tiago/Desktop/repos/GEDI-H3/tmp/tmp/l2a/h3_03=838041fffffffff')
# table = dataset.read()
# df = table.to_pandas()

df = gpd.read_parquet(f)
df

866 ms ± 87.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [31]:
df.to_parquet('../tmp/singleton.parquet', compression='zstd')

In [ ]:
df['geometry'] = gpd.GeoSeries.from_wkb(df.geometry)
df

In [ ]:
from dask.distributed import Client
import dask.dataframe
import dask_geopandas
client = Client(processes=True, threads_per_worker=1, n_workers=2)
client

In [ ]:
f = soc_file_tree(GH3_DEFAULT_SOC_DIR, to_list=True)
dat_col = 'delta_time'
lat_col = 'lat_lowestmode'
lon_col = 'lon_lowestmode'
ddf = dask_h5_merged(f, {'L2A':['rh','lon_lowestmode','lat_lowestmode','delta_time'], 'L4A': ['agbd'], 'L4C': ['wsci']}, by_beam=True)
ddf

In [ ]:
ddf['datetime'] = dask.dataframe.to_datetime(ddf[dat_col] + GEDI_START_DATE.timestamp(), unit='s')
ddf['year'] = ddf.datetime.dt.year

ddf = ddf.map_partitions(h3_index_df)
h3_tiles = ['838041fffffffff', '83804cfffffffff', '83804efffffffff']
ddf = ddf[ddf.h3_03.isin(h3_tiles)]

ddf = dask_geopandas.from_dask_dataframe(ddf, geometry=dask_geopandas.points_from_xy(ddf, x=lon_col, y=lat_col, crs='EPSG:4326'))
ddf

In [ ]:
ddf.head()

In [ ]:
ddf.to_parquet('/home/tiago/Desktop/repos/GEDI-H3/tmp/tmp/parquet_test', partition_on=['h3_03'])